# 01 · Define & Explore — serine-hydrolase/PETase catalysis, the triad, and the thermostability bottleneck

**Standard slot:** *define & explore.* **For Project 19 this means:** understand de novo enzyme
design and PET hydrolysis (a serine-hydrolase reaction), then **construct the theozyme** (the
Ser-His-Asp triad + oxyanion hole around an ester transition state) and run a mock theozyme→scaffold
hello-world (D0). The emphasis throughout is **thermostability** — the real bottleneck for PET.

Run `00_setup.ipynb` first in this session.

## Why PET hydrolases — and why thermostability is the bottleneck
PET (polyethylene terephthalate) is a hugely-produced plastic that is barely recycled. **PET
hydrolases** (IsPETase, the engineered cutinase LCC) cut the PET ester backbone back to its monomers
(MHET / TPA + ethylene glycol), enabling **circular chemistry**. Two facts shape this project:
- **The chemistry is known.** PET hydrolysis is a textbook **serine-hydrolase** reaction — the same
  **Ser-His-Asp triad + oxyanion hole** as lipases, esterases, and cutinases. Unlike the Kemp
  elimination (Project 18), this reaction *has* natural counterparts.
- **Stability is the hard part.** PET only becomes accessible to enzymes near its glass transition
  (~65-70 °C), and **wild-type IsPETase falls apart there.** The breakthrough (Tournier 2020) was
  *thermostabilising* the enzyme. So this campaign is decided by **thermostability**, not catalytic
  novelty.

The honest history: de novo enzymes are rarely active first try (<5% without directed evolution), and
even when geometry is right, the thermostability ↔ activity trade-off bites.

## The theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal set of catalytic functional groups placed
around the **transition state** (here, the tetrahedral intermediate of ester hydrolysis):

| Role | Residue(s) | Job in the TS |
|------|-----------|----------------|
| catalytic Ser | Ser (OG) | nucleophile — attacks the ester carbonyl carbon |
| catalytic His | His (NE2) | general base — deprotonates Ser-OH to activate it |
| catalytic Asp | Asp / Glu (OD) | orients/protonates His (the charge-relay) |
| oxyanion hole | 2× backbone-NH (or Ser-OG) | stabilise the developing oxyanion of the tetrahedral intermediate |

The **oxyanion hole is easy to forget and decisive** — without it the tetrahedral intermediate is not
stabilised and there is no catalysis even with a perfect triad. You **construct** this from the
literature and/or a QM transition-state model — it is a teaching template
(`data/inputs/theozyme_def.txt`), **not** fabricated data. Place groups around the **TS**, not the
ground-state ester.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme(reaction)` → a functional-group geometry spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/theozyme_def.txt`
(and in `build_theozyme`) with real, cited values during P1. This follows the **enzyme-family
template** (Project 18, Kemp): the reaction (ester hydrolysis) and emphasis (thermostability) differ;
the workflow is the same.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("ester_hydrolysis")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from literature/QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:16s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nCatalytic residues to FIX during sequence design (triad + oxyanion hole):")
print(" ", theo.catalytic_residue_ids())

## A first mock scaffold + sequence + thermostability proxy (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the motif;
`ligandmpnn_fix_catalytic(...)` (mock) designs sequences with the catalytic triad fixed;
`thermostability_md(...)` (mock) returns the project's headline stability proxy. **Every number here
is SYNTHETIC** — this only proves the plumbing runs anywhere. Switch to the real backends
(RFdiffusion2/Riff-Diff + OpenMM on an A100; LigandMPNN CPU-fast) in `02_generate.ipynb` / `04`.

In [ ]:
from enzyme_tools import (scaffold_motif, ligandmpnn_fix_catalytic,
                          catalytic_geometry_rmsd, thermostability_md, dock_substrate)

scaffolds = scaffold_motif(theo, n=5, method="mock", track="de_novo")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_fix_catalytic(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_catalytic_roles']})")

cg = catalytic_geometry_rmsd(None, theo)         # mock, SYNTHETIC
thermo = thermostability_md(None, ns=20.0)       # mock, SYNTHETIC — the project emphasis
dock = dock_substrate(None, theo.substrate)      # mock, SYNTHETIC — pocket accessibility
print(f"\ncatalytic_geometry_rmsd (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print(f"thermostability proxy (mock, SYNTHETIC): catalytic_rmsf={thermo['catalytic_rmsf']} A, "
      f"melting_proxy={thermo['melting_proxy']} (higher=more stable)")
print(f"PET-mimic docking (mock, SYNTHETIC): in_pocket={dock['pose_in_pocket']}, "
      f"oriented_to_Ser={dock['oriented_to_ser']}")
print("\nNOTE: these are placeholder numbers. The real campaign is in notebook 02; the "
      "thermostability ranking is in notebook 04.")

## The metrics that decide a PET-hydrolase design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the triad + oxyanion hole* | the geometry is correct |
| **catalytic_geom_rmsd** | **< 0.5 Å** | predicted catalytic atoms vs the theozyme | **activity** (the design can still be dead) |
| **thermostability proxy** (MD) | *rank, not a hard cutoff* | RMSF + melting-proxy — survives heat? | **a measured Tm** (DSF decides) |

The last two rows are the point of this project — and the last column is the message to never forget:
**in-silico catalytic geometry + an MD stability proxy do not guarantee a working, thermostable
enzyme.** Only an activity assay (pNP-ester / PET-film) + DSF decide (notebook 05).

## D0 checklist
- [ ] Half-page on de novo enzyme design + the honest hit-rate history + **why thermostability is the PET bottleneck**.
- [ ] 1-page problem statement with **measurable** success criteria (incl. a thermostability target) + the controls you'll need.
- [ ] Theozyme spec started in `data/inputs/theozyme_def.txt` (triad + oxyanion hole; replace the PLACEHOLDERs, cite sources).
- [ ] Reproduced mock hello-world (triad + oxyanion-hole spec + a mock scaffold record + a thermostability proxy).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the triad into stable folds and run LigandMPNN with the catalytic triad fixed.